In [189]:
import pandas as pd
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split

In [190]:
df = pd.read_csv("/home/havin-benoy/Desktop/Pyspiders/ml/Datasets/loan_data.csv")
df.head()

,person_age,person_gender,person_education,person_income,person_emp_exp,person_home_ownership,loan_amnt,loan_intent,loan_int_rate,loan_percent_income,cb_person_cred_hist_length,credit_score,previous_loan_defaults_on_file,loan_status
0,22.0,female,Master,71948.0,0,RENT,35000.0,PERSONAL,16.02,0.49,3.0,561,No,1
1,21.0,female,High School,12282.0,0,OWN,1000.0,EDUCATION,11.14,0.08,2.0,504,Yes,0
2,25.0,female,High School,12438.0,3,MORTGAGE,5500.0,MEDICAL,12.87,0.44,3.0,635,No,1
3,23.0,female,Bachelor,79753.0,0,RENT,35000.0,MEDICAL,15.23,0.44,2.0,675,No,1
4,24.0,male,Master,66135.0,1,RENT,35000.0,MEDICAL,14.27,0.53,4.0,586,No,1


In [191]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 45000 entries, 0 to 44999
Data columns (total 14 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   person_age                      45000 non-null  float64
 1   person_gender                   45000 non-null  str    
 2   person_education                45000 non-null  str    
 3   person_income                   45000 non-null  float64
 4   person_emp_exp                  45000 non-null  int64  
 5   person_home_ownership           45000 non-null  str    
 6   loan_amnt                       45000 non-null  float64
 7   loan_intent                     45000 non-null  str    
 8   loan_int_rate                   45000 non-null  float64
 9   loan_percent_income             45000 non-null  float64
 10  cb_person_cred_hist_length      45000 non-null  float64
 11  credit_score                    45000 non-null  int64  
 12  previous_loan_defaults_on_file  45000 non-n

In [192]:
x = df.drop(columns="loan_status")
y = df["loan_status"]

In [193]:
x_train, x_test, y_train, y_test = train_test_split(x,y,train_size=0.8,random_state=42)

In [194]:
# To get the categorical values from the dataset. 
# specifically from the the input feature columns 
categorical_cols = x.select_dtypes(include = 'object').columns


print(categorical_cols)

Index(['person_gender', 'person_education', 'person_home_ownership',
       'loan_intent', 'previous_loan_defaults_on_file'],
      dtype='str')


/tmp/ipykernel_194744/1054614932.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = x.select_dtypes(include = 'object').columns


In [195]:
print(categorical_cols.nunique())   
# Gives the number of unique values in each columns of the categorical columns 


5


Note:

- That means there are 5 categorical columns in the dataset


In [196]:
onehot_encoder = OneHotEncoder(
    sparse_output= False,    # True or False
    handle_unknown='ignore' # 'ignore' or 'error'
)

Note:

- there are variables we need to pass for the OneHotEncoder()
    1. sparse_output :
        - if we pass True for this , then a compressed sparse_matrix version of full output is prodced. It might only contain the value 1's , and not zeros
        - But we need the full output , so that we can assign it into the columns.
        - so we pass 'False' to get full encoded values rather than a compressed representation.
    
    2. handle_unknown:
        - it is for handling unknown values.
        - based on the object (onehot_encoder), we will first do fit_transform(x_train[categorical_cols])
        - here the unique values in all the categorical columns will be taken , sorted and make into columns.
        - this unique value information regarding the passed categorical columns are stored in the object (onehot_encoder)
        - if during the testing any new unseen unique value is passed inside the categorical column , the obj.transform() can raise an error or simply give 0 to all the existing encoded column rows of the corresponding to the row of unexpected value.
        - 
        - example: given below is a column in a dataframe 

            bool_col
             Yes
             No
             NO
             Yes

        - then after obj.fit_transform(x_train[[bool_col]]) it will become :

            bool_col_Yes            bool_col_No
                1                       0
                0                       1
                0                       1
                1                       0

        - this information of the unique values (bool_col_Yes, bool_col_No ) is stored in the obj
        - now consider the below unseen data:

                unseen_bool_col
                    Ysss
                    No

        - see that the first value 'Ysss' was not present during the fit_transform()
        - so that information will not the be present in the obj
        - so when we call obj.transform(unseen_bool_col), the transform() will only fill calculate the corresponding column values of the existing columns as per the obj
        - the obj contains the columns 'bool_col_Yes' and 'bool_col_No'
        - so if we pass handle_unknown = 'error', we will get error when we run obj.transform(unseen_bool_col)
        - but if we pass handle_unknown = 'ignore', we will not get error and the following output will be generated 
            bool_col_Yes            bool_col_No
                0                       0
                0                       1
        - see that the full row corresponding to 'Ysss' is turned to be 0.
        - because no 'Ysss' is not equal to 'Yes' or 'No'




In [197]:
encoded_values = onehot_encoder.fit_transform(x_train[categorical_cols])
encoded_values

array([[1., 0., 0., ..., 0., 1., 0.],
       [0., 1., 0., ..., 1., 0., 1.],
       [1., 0., 0., ..., 0., 0., 1.],
       ...,
       [1., 0., 1., ..., 1., 0., 1.],
       [0., 1., 0., ..., 0., 1., 0.],
       [1., 0., 0., ..., 0., 0., 1.]], shape=(36000, 19))

### Note:
- 19 means the sum of unique values of all the categorical columns we passed
- I.e that many columns are generated after the OneHotEncoding()


### Note :

- The methods fit_transform() and transform() will only accept 2d arrays
- so if the code as like below , it will generate ValeError
    > encoded_values = onehot_encoder.fit_transform(x_train["person_education"])

- so even if we pass a single column , we need to mention it as 2d. Like below:
    > encoded_values = onehot_encoder.fit_transform(x_train[["person_education"]])

### Note:
- Workflow of fit_transform():
    1. select the unique values from all the columns
    2. sorts by ascending order
    3. creates new columns

- workflow of transform():
    1. encodes the values
        - assigns 1 if the label is present in the row 
        - else assigns 0 


In [198]:
encoded_cols = onehot_encoder.get_feature_names_out()
encoded_cols

# it gives us all the column names of the newly generated columns afte rthe OneHotEncoding
# It follows the pattern of ColumnName_UniqueValueName

array(['person_gender_female', 'person_gender_male',
       'person_education_Associate', 'person_education_Bachelor',
       'person_education_Doctorate', 'person_education_High School',
       'person_education_Master', 'person_home_ownership_MORTGAGE',
       'person_home_ownership_OTHER', 'person_home_ownership_OWN',
       'person_home_ownership_RENT', 'loan_intent_DEBTCONSOLIDATION',
       'loan_intent_EDUCATION', 'loan_intent_HOMEIMPROVEMENT',
       'loan_intent_MEDICAL', 'loan_intent_PERSONAL',
       'loan_intent_VENTURE', 'previous_loan_defaults_on_file_No',
       'previous_loan_defaults_on_file_Yes'], dtype=object)

In [199]:
# To convert the result into a dataframe
pd.DataFrame(data = encoded_values,columns = encoded_cols)

,person_gender_female,person_gender_male,person_education_Associate,person_education_Bachelor,person_education_Doctorate,person_education_High School,person_education_Master,person_home_ownership_MORTGAGE,person_home_ownership_OTHER,person_home_ownership_OWN,person_home_ownership_RENT,loan_intent_DEBTCONSOLIDATION,loan_intent_EDUCATION,loan_intent_HOMEIMPROVEMENT,loan_intent_MEDICAL,loan_intent_PERSONAL,loan_intent_VENTURE,previous_loan_defaults_on_file_No,previous_loan_defaults_on_file_Yes
0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0
1,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0
2,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0
3,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
4,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
35995,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0
35996,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
35997,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0
35998,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0


In [ ]:
# Transforming the test data
onehot_encoder.transform(x_test[categorical_cols])


array([[0., 1., 1., ..., 0., 1., 0.],
       [0., 1., 1., ..., 0., 0., 1.],
       [1., 0., 0., ..., 0., 1., 0.],
       ...,
       [1., 0., 0., ..., 0., 1., 0.],
       [0., 1., 1., ..., 0., 1., 0.],
       [1., 0., 0., ..., 0., 1., 0.]], shape=(9000, 19))

Note :
- transform() will accept 2d
- returns the values based on the sparse_output

- as we already discussed.
- if handle_unknown = 'error'
    - for unknown labels it throws error(valueError)
- if handle_unknown = 'ignore'
    - Assigns zero for unknown labels

## Creating new unseen data to test this onehotencoding
 - here we are filling the values of the categorical columns with unseen future values
 - in future some values might be due to some error could contain error data as well.
 - we already discussed how we handle it via the handle_unknown parameter
 - example in the last column "previous_loan_defaults_on_file" contains the values 'Yesss' and 'Noo' which are not in the object (onehot_encoder).
 - it only knows ['Yes','No']
 - so change the handle_unknown parameter as 'error' and 'ignore' and see how it is reflected in the output.
 - in the output , there will be only columnName_Yes and columnName_No , there will not be columnName_Yesss or columnName_Noo because transform() will not create the column header , it can only fill the values. so the rows corresponding to the Yes and No will be filled with 0.

In [202]:
new_data = pd.DataFrame({
    'person_gender': ['female','male'],
    'person_education':['Bachelor','UG'],
    "person_home_ownership": ['MORTGAGE','RENT'],
    'loan_intent': ['PERSONAL','PERSONAL'],
    'previous_loan_defaults_on_file': ['Yesss','Noo']
})
display(new_data)

,person_gender,person_education,person_home_ownership,loan_intent,previous_loan_defaults_on_file
0,female,Bachelor,MORTGAGE,PERSONAL,Yesss
1,male,UG,RENT,PERSONAL,Noo


In [203]:
unseen_vals = onehot_encoder.transform(new_data)

In [204]:
pd.DataFrame(data = unseen_vals, columns = encoded_cols)

,person_gender_female,person_gender_male,person_education_Associate,person_education_Bachelor,person_education_Doctorate,person_education_High School,person_education_Master,person_home_ownership_MORTGAGE,person_home_ownership_OTHER,person_home_ownership_OWN,person_home_ownership_RENT,loan_intent_DEBTCONSOLIDATION,loan_intent_EDUCATION,loan_intent_HOMEIMPROVEMENT,loan_intent_MEDICAL,loan_intent_PERSONAL,loan_intent_VENTURE,previous_loan_defaults_on_file_No,previous_loan_defaults_on_file_Yes
0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
1,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0


# Note:
- we can see the last two columns are previous_loan_defaults_on_file_No and previous_loan_defaults_on_file_Yes and both are filled with 0.
- because 'Yesss' is not matching with both Yes or No
- also 'Noo' is not matching with both Yes or No